# Exploratory Data Analysis of Solar PV Production

## 1. Introduction

Solar photovoltaic (PV) systems convert solar energy into electricity. Understanding their electricity production patterns is important for renewable energy planning, system performance assessment, and energy forecasting.

This project investigates historical solar electricity production data from photovoltaic installations operated by the City of Calgary, Canada.

The analysis will use Python to inspect, clean, preprocess, and visualize the dataset. Particular attention will be given to temporal production patterns, differences between installations, data quality, and the potential use of historical production data for energy forecasting.

### Project objectives

- Understand the structure and characteristics of the dataset.
- Identify and address data quality issues.
- Analyze temporal patterns in solar electricity production.
- Compare production across photovoltaic installations where appropriate.
- Identify insights relevant to renewable energy assessment and forecasting.

### Dataset

- **Dataset name:** Solar Production (Photovoltaic Output Data)
- **File:** Solar_Energy_Production.csv
- **Main variable of interest:** Energy production (kWh)

### Research question

How does photovoltaic electricity production vary over time and across installations, and what can historical production data reveal about generation patterns and forecasting potential?

In [1]:
# Import the libraries required for data analysis and visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure the default visualization style
sns.set_theme(style="whitegrid")

# Display plots directly inside the notebook
%matplotlib inline

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Data Loading and Inspection

The dataset is stored in CSV format. The pandas library will be used to import the file into a DataFrame, which allows the data to be examined and processed.

The initial inspection will verify that the file has been loaded correctly before assessing its structure, variable types, and statistical characteristics.

In [2]:
# Load the solar production dataset

from pathlib import Path

file_path = Path("../data/Solar_Energy_Production.csv")

df = pd.read_csv(file_path)

# Confirm that the dataset was loaded
print("Dataset loaded successfully.")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

# Display the first five records
display(df.head())

Dataset loaded successfully.
Number of rows: 258,423
Number of columns: 8


,name,id,address,date,kWh,public_url,installationDate,uid
0,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 08:00:00 AM,1.130,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 08:00:00
1,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 09:00:00 AM,2.340,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 09:00:00
2,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 10:00:00 AM,3.656,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 10:00:00
3,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 11:00:00 AM,4.577,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 11:00:00
4,Calgary Fire Hall Headquarters,314106,1212 42 AV SE,2017/09/11 12:00:00 PM,6.506,https://monitoring.solaredge.com/solaredge-web...,2016/11/07,3141062017-09-11 12:00:00


### 2.1 Initial Dataset Inspection

The dataset will be examined using its dimensions, column names, data types, and descriptive statistics. This step helps identify the variables available for analysis and highlights potential issues that may require preprocessing.

In [3]:
# Inspect column names, data types, and non-null counts
df.info()

# Generate descriptive statistics
display(df.describe(include="all"))

# Check for missing values in each column
display(df.isnull().sum())

# Check for exact duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258423 entries, 0 to 258422
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   name              258423 non-null  object 
 1   id                258423 non-null  int64  
 2   address           258423 non-null  object 
 3   date              258423 non-null  object 
 4   kWh               258423 non-null  float64
 5   public_url        258423 non-null  object 
 6   installationDate  258423 non-null  object 
 7   uid               258423 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 15.8+ MB


,name,id,address,date,kWh,public_url,installationDate,uid
count,258423,258423.000000,258423,258423,258423.000000,258423,258423,258423
unique,11,NaN,11,36192,NaN,22,11,258423
top,Southland Leisure Centre,NaN,2000 SOUTHLAND DR SW,2017/12/05 04:00:00 PM,NaN,https://monitoringpublic.solaredge.com/solared...,2015/09/01,3141062017-09-11 08:00:00
freq,31590,NaN,31590,11,NaN,24533,31590,1
mean,NaN,372581.033132,NaN,NaN,38.059762,NaN,NaN,NaN
std,NaN,130344.544572,NaN,NaN,71.598031,NaN,NaN,NaN
min,NaN,164440.000000,NaN,NaN,0.000000,NaN,NaN,NaN
25%,NaN,314106.000000,NaN,NaN,1.370000,NaN,NaN,NaN
50%,NaN,331901.000000,NaN,NaN,8.034000,NaN,NaN,NaN
75%,NaN,551172.000000,NaN,NaN,35.876000,NaN,NaN,NaN


name                0
id                  0
address             0
date                0
kWh                 0
public_url          0
installationDate    0
uid                 0
dtype: int64

Number of duplicate rows: 0


### 2.2 Data Quality Assessment

The initial inspection indicates that the dataset contains no
missing values in the displayed columns and no exact duplicate
rows. However, further checks are required to assess timestamp
validity, chronological consistency, and gaps in the recorded
production data.

The next step is to convert the date columns to appropriate
datetime formats and examine the range and distribution of
the recorded energy production values.

In [4]:

# Inspect the date columns and energy production values

print("Column names:")
print(df.columns.tolist())

# Convert date columns to datetime format
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["installationDate"] = pd.to_datetime(
    df["installationDate"], errors="coerce"
)

# Check whether any dates failed to parse
print("\nInvalid or missing production dates:", df["date"].isna().sum())
print("Invalid or missing installation dates:",
      df["installationDate"].isna().sum())

# Examine the date range
print("\nProduction date range:")
print("Earliest:", df["date"].min())
print("Latest:", df["date"].max())

# Examine the energy production variable
print("\nEnergy production summary:")
display(df["kWh"].describe())

print("\nNegative energy readings:", (df["kWh"] < 0).sum())
print("Zero energy readings:", (df["kWh"] == 0).sum())


Column names:
['name', 'id', 'address', 'date', 'kWh', 'public_url', 'installationDate', 'uid']

Invalid or missing production dates: 0
Invalid or missing installation dates: 0

Production date range:
Earliest: 2015-09-01 14:30:00
Latest: 2023-03-16 19:00:00

Energy production summary:


count    258423.000000
mean         38.059762
std          71.598031
min           0.000000
25%           1.370000
50%           8.034000
75%          35.876000
max         514.462000
Name: kWh, dtype: float64


Negative energy readings: 0
Zero energy readings: 1176
